In [1]:
import torch
import numpy as np

In [2]:
class biRNN(torch.nn.Module):
    def __init__(self, vocab_size: int, embedding_size:int, hidden_size: int, output_size: int, layers: int = 1, dropout: float = 0.2):
        super().__init__()

        self.embedding = torch.nn.Embedding(vocab_size, embedding_size)

        self.layers = layers
        self.hidden_size = hidden_size
        self.output_size = output_size

        self.Wxh_forward = torch.nn.ParameterList()
        self.Whh_forward = torch.nn.ParameterList()
        self.bh_forward = torch.nn.ParameterList()

        self.Wxh_backward = torch.nn.ParameterList()
        self.Whh_backward = torch.nn.ParameterList()
        self.bh_backward = torch.nn.ParameterList()
        
        self.layer_norms = torch.nn.ModuleList() 

        for i in range(layers):
            current_input_size = embedding_size if i == 0 else hidden_size * 2

            self.Wxh_forward.append(torch.nn.Parameter(torch.randn(current_input_size, hidden_size) * 0.01))
            self.Whh_forward.append(torch.nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01))
            self.bh_forward.append(torch.nn.Parameter(torch.zeros(hidden_size)))

            self.Wxh_backward.append(torch.nn.Parameter(torch.randn(current_input_size, hidden_size) * 0.01))
            self.Whh_backward.append(torch.nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01))
            self.bh_backward.append(torch.nn.Parameter(torch.zeros(hidden_size)))
            
            self.layer_norms.append(torch.nn.LayerNorm(hidden_size * 2))

        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(hidden_size * 2, hidden_size),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden_size, output_size)
        )

    def forward(self, x):

        x = self.embedding(x)
        
        batch_size = x.size(0)
        seq_len = x.size(1)

        for layer in range(self.layers):
            h_forward = torch.zeros(batch_size, self.hidden_size, device=x.device)
            h_backward = torch.zeros(batch_size, self.hidden_size, device=x.device)

            forward_output = []
            backward_output = []

            for t in range(seq_len):
                xt = x[:, t, :]
                h_forward = torch.tanh(xt @ self.Wxh_forward[layer] + h_forward @ self.Whh_forward[layer] + self.bh_forward[layer])
                forward_output.append(h_forward)

            for t in reversed(range(seq_len)):
                xt = x[:, t, :]
                h_backward = torch.tanh(xt @ self.Wxh_backward[layer] + h_backward @ self.Whh_backward[layer] + self.bh_backward[layer])
                backward_output.append(h_backward)

            backward_output.reverse()

            f_out_tensor = torch.stack(forward_output, dim=1)
            b_out_tensor = torch.stack(backward_output, dim=1)

            x = torch.cat([f_out_tensor, b_out_tensor], dim=-1)
            
            x = self.layer_norms[layer](x)

        y = self.classifier(x)

        return y

In [5]:
import nltk

nltk.data.path.append(r"C:\Users\offsh\Documents\project\pos\data\nltk_data")

from nltk.corpus import treebank
from collections import defaultdict

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

In [6]:
tagged_sentences = treebank.tagged_sents(tagset='universal')

In [7]:
split_idx = int(len(tagged_sentences) * 0.8)

train_data = tagged_sentences[:split_idx]
test_data = tagged_sentences[split_idx:]

len(train_data), len(test_data)

(3131, 783)

In [8]:
pad_idx = 0
ukn_idx = 1

In [9]:
word2idx = defaultdict(lambda: ukn_idx)
word2idx['<PAD>'] = pad_idx
word2idx['<UNK>'] = ukn_idx

In [10]:
tag2idx = {'<PAD>': pad_idx}

In [11]:
for sentence in train_data:
    for word, tag in sentence:
        word = word.lower()
        if word not in word2idx:
            word2idx[word] = len(word2idx)
        if tag not in tag2idx:
            tag2idx[tag] = len(tag2idx)

idx2tag = {idx: tag for tag, idx in tag2idx.items()}

In [14]:
len(word2idx), len(tag2idx)

(10153, 13)

In [15]:
class POSDataset(Dataset):
    def __init__(self, data, word2idx, tag2idx):
        self.data = data
        self.word2idx = word2idx
        self.tag2idx = tag2idx

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sentence = self.data[idx]
        word_ids = [self.word2idx[word.lower()] for word, tag in sentence]
        tag_ids = [self.tag2idx[tag] for word, tag in sentence]
        return torch.tensor(word_ids), torch.tensor(tag_ids)

In [16]:
def collate_fn(batch):
    words, tags = zip(*batch)
    words_padded = pad_sequence(words, batch_first=True, padding_value=pad_idx)
    tags_padded = pad_sequence(tags, batch_first=True, padding_value=pad_idx)
    return words_padded, tags_padded

In [17]:
train_loader = DataLoader(POSDataset(train_data, word2idx, tag2idx), batch_size=32, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(POSDataset(test_data, word2idx, tag2idx), batch_size=32, collate_fn=collate_fn)

In [20]:
words, tags = next(iter(train_loader))
w, t = next(iter(test_loader))

print(words.shape)
print(tags.shape)
print(w.shape)
print(t.shape)

torch.Size([32, 67])
torch.Size([32, 67])
torch.Size([32, 48])
torch.Size([32, 48])


In [21]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [22]:
model = biRNN(
    vocab_size=len(word2idx), 
    embedding_size=64, 
    hidden_size=128, 
    output_size=len(tag2idx), 
    layers=1
).to(device)

In [23]:
criterion = torch.nn.CrossEntropyLoss(ignore_index = pad_idx)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

In [24]:
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for words, tags in train_loader:
        words, tags = words.to(device), tags.to(device)
        
        optimizer.zero_grad()
        logits = model(words) 
        
        logits = logits.view(-1, logits.shape[-1])
        tags = tags.view(-1)
        
        loss = criterion(logits, tags)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")

model.eval()
correct = 0
total_words = 0

with torch.no_grad():
    for words, tags in test_loader:
        words, tags = words.to(device), tags.to(device)
        logits = model(words)
        
        predictions = torch.argmax(logits, dim=-1)
        
        mask = (tags != pad_idx)
        correct += (predictions[mask] == tags[mask]).sum().item()
        total_words += mask.sum().item()

print(f"\nTest Accuracy: {100 * correct / total_words:.2f}%")

Epoch 1/10 | Loss: 0.6787
Epoch 2/10 | Loss: 0.3416
Epoch 3/10 | Loss: 0.2250
Epoch 4/10 | Loss: 0.1536
Epoch 5/10 | Loss: 0.1065
Epoch 6/10 | Loss: 0.0751
Epoch 7/10 | Loss: 0.0545
Epoch 8/10 | Loss: 0.0442
Epoch 9/10 | Loss: 0.0347
Epoch 10/10 | Loss: 0.0302

Test Accuracy: 92.61%
